# Пространственные отношения


В предыдущем разделе мы научились изменять геометрию объектов и создавать новые пространственные данные. Однако во многих задачах важно не только преобразовывать геометрию, но и анализировать, **как объекты расположены относительно друг друга**.

Пространственные отношения позволяют определить, пересекаются ли объекты, находятся ли они внутри других объектов или касаются ли они друг друга. Эти операции лежат в основе многих задач пространственного анализа, таких как:

- поиск объектов в заданной зоне;
- определение доступности;
- фильтрация данных по пространственному признаку.

В GeoPandas для этого используются специальные методы, называемые **пространственными предикатами** (spatial predicates), например: `intersects`, `within`, `contains`.

В этом разделе мы рассмотрим, как применять пространственные отношения для анализа и фильтрации геоданных.


## 0. Импорт библиотек и подготовка данных


### 0.1 Импорт библиотек


In [ ]:
import pandas as pd  
import geopandas as gpd
import osmnx as ox

### 0.2 Подготовка данных


Загрузим границу Центрального района Санкт-Петербурга из OpenStreetMap


In [ ]:
area_name = "Центральный район, Санкт-Петербург"
admin_border = ox.geocode_to_gdf(area_name)

admin_border.explore(tiles="cartodbpositron")

Прочитаем данные о театрах Санкт-Петербурга из CSV-файла и создадим на их основе `GeoDataFrame`.

Данные можно найти в нашем [репозитории](https://github.com/bella-mir/geoPython/tree/main/chapters/module_3/data):

- **spb_theaters.csv** — данные о театрах Санкт-Петербурга.  
  Источник: [Портал открытых данных Санкт-Петербурга](https://data.gov.spb.ru/)


In [ ]:
theaters_csv = pd.read_csv('./data/spb_theaters.csv')
theaters_csv= theaters_csv.dropna(subset=['longitude', 'latitude']) 

theaters_gpd = gpd.GeoDataFrame(
    theaters_csv,
    geometry=gpd.points_from_xy(theaters_csv['longitude'], theaters_csv['latitude']),
    crs = 4326
)

theaters_gpd.explore(tiles="cartodbpositron")

## 1. Топологические (пространственные) отношения

**Топологические отношения** (или **пространственные отношения**) — это операции, позволяющие определить взаимное расположение двух геометрических объектов (точек, линий, полигонов и т.д.) в пространстве.

Они отвечают на вопросы вида:
«Пересекаются ли объекты?»,
«Находится ли один объект внутри другого?»,
«Соприкасаются ли их границы?» и т.п.

Знание топологических отношений используется для выполнения пространственных запросов (spatial queries), пространственных соединений (spatial join), фильтрации данных и анализа взаимного расположения объектов.

### 1.1 Основные топологические отношения

Ниже приведены основные бинарные отношения между геометриями:

| **Отношение**                            | **Описание (условие истинности)**                                                                                                                                                                                        |
| ---------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| **intersects** (_пересекается_)          | **Имеют общую точку** — Геометрии пересекаются, если у них есть хотя бы одна общая точка (пересечение границ или внутренних областей). Эквивалентно отрицанию disjoint.                                                  |
| **disjoint** (_раздельны_)               | **Не имеют общих точек** — Геометрии полностью разделены в пространстве и не имеют ни одной общей точки.                                                                                                                 |
| **within** (_внутри_)                    | **A внутри B** — Геометрия A полностью находится внутри геометрии B. Все точки A лежат во внутренней области B. (Эквивалентно B.contains(A)).                                                                            |
| **contains** (_содержит_)                | **A содержит B** — Геометрия A полностью содержит геометрию B, при этом хотя бы одна точка B лежит строго внутри A. (Эквивалентно B.within(A)).                                                                          |
| **touches** (_касается_)                 | **Касание границ** — Геометрии имеют общие точки только на границе, но их внутренние области не пересекаются.                                                                                                            |
| **overlaps** (_перекрывается_)           | **Частичное перекрытие** — Геометрии частично совпадают по области, но ни одна полностью не содержит другую (обычно для объектов одинаковой размерности, например полигон–полигон).                                      |
| **crosses** (_пересекается (с выходом)_) | **Пересечение с изменением размерности** — Геометрии пересекаются внутренними точками, при этом результат пересечения имеет меньшую размерность (например, линия пересекает полигон или две линии пересекаются в точке). |


### 1.2 Топологические отношения в GeoPandas

В GeoPandas топологические отношения реализованы в виде методов, применяемых к геометриям и возвращающих логические значения (`True` или `False`).

Такие методы проверяют выполнение определённого пространственного отношения для каждой пары геометрий (например, `within`, `contains`, `intersects` и др.).

На практике это означает, что можно выполнять фильтрацию и анализ данных на основе взаимного расположения объектов.

Например, проверим, какие театры находятся внутри Центрального района Санкт-Петербурга.


In [ ]:
theaters_gpd.within(admin_border.geometry.iloc[0])

Результатом будет серия значений `True/False`, где:

- `True` — объект удовлетворяет условию (находится внутри);
- `False` — не удовлетворяет.

Рассмотрим, как пространственные предикаты применяются на практике для фильтрации геоданных.


## 2. Фильтрация объектов по пространственному признаку


Одно из распространённых действий в пространственном анализе – отбор объектов, удовлетворяющих определённому пространственному условию. Например, можно захотеть **выбрать все точки (объекты) внутри заданного полигона**. Это называется пространственной фильтрацией (фильтрация по местоположению).

Давайте отфильтруем все театры, которые находятся в Центральном районе Санкт-Петербурга


In [ ]:
theaters_center = theaters_gpd[
    theaters_gpd.geometry.within(admin_border.geometry.iloc[0])
]

In [ ]:
theaters_center.explore(tiles='CartoDB positron')

На карте видно, что отобраны только те театры, которые находятся в Центральном районе Санкт-Петербурга.


## Итог

В этом разделе мы познакомились с **топологическими (пространственными) отношениями** и научились применять их для анализа геоданных.

Мы узнали:

- что такое топологические отношения и какие их типы существуют;
- как использовать соответствующие методы в GeoPandas;
- как фильтровать объекты на основе их взаимного пространственного расположения.

В рассмотренных примерах мы использовали топологические отношения для фильтрации данных. Однако при работе с несколькими наборами данных часто требуется не только отобрать объекты, но и объединить их атрибутивную информацию.

Для этого используется операция **пространственного соединения** (_spatial join_), которую мы рассмотрим в следующем разделе.
